In [72]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_nuclei_labels_output_dir,
    load_precomputed_nuclei_labels_if_available,
)
from utils.segmentation import predict_nuclei_labels

from utils.inference import predict_tiled_unet

import tifffile
import napari
import torch

In [73]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 2

In [74]:
# Iterate through the .lif files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [75]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (115, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (116, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (135, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (107, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (118, 4, 256, 1024)


In [76]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

[<Image layer 'edCerulean_CTRL' at 0x1ee09580a30>,
 <Image layer 'edCitrine_FRET' at 0x1ee095a3e20>,
 <Image layer 'edCitrine_CTRL' at 0x1ee095ebd60>,
 <Image layer 'brightfield' at 0x1ee09641d80>]

In [77]:
import numpy as np
from skimage.filters import difference_of_gaussians

def _normalize_percentile(image: np.ndarray, pmin: int = 1, pmax: int = 99) -> np.ndarray:
    """
    Normalize an image to the [0,1] range based on percentile clipping.

    Args:
        image (np.ndarray): The input image to be normalized.
        pmin (int, optional): Lower percentile for normalization. Defaults to 1.
        pmax (int, optional): Upper percentile for normalization. Defaults to 99.

    Returns:
        np.ndarray: The normalized image with values in [0, 1].
    """
    vmin, vmax = np.percentile(image, (pmin, pmax))
    image = np.clip(image, vmin, vmax)  # clip outlier intensities
    if vmax > vmin:
        image = (image - vmin) / (vmax - vmin)  # rescale to [0, 1]
    else:
        image = image * 0  # avoid division by zero; returns zeros
    return image

def _normalize_full_range(image: np.ndarray) -> np.ndarray:
    """
    Normalize an image to the [0,1] range using full 0-100 percentiles.

    Args:
        image (np.ndarray): The input image to be normalized.

    Returns:
        np.ndarray: The normalized image.
    """
    return _normalize_percentile(image, pmin=0, pmax=100)

def simulate_fluo_from_bf(
    lif_image: np.ndarray, 
    markers: tuple, 
    low_sigma: float = 1, 
    high_sigma: float = 2
) -> np.ndarray:
    """
    Simulate a fluorescent cell boundary channel from a brightfield image.

    Args:
        lif_image (np.ndarray): 4D image data, assumed shape (C, Z, Y, X) or (C, ...).
        markers (tuple): Channel descriptors linking channel names to indices.
        low_sigma (float, optional): Lower sigma for DoG. Defaults to 1.
        high_sigma (float, optional): Higher sigma for DoG. Defaults to 2.

    Returns:
        np.ndarray: Simulated boundary image for UNet3D input.
    """
    # Find the brightfield channel index in the user-defined markers
    bf_index = None
    for marker in markers:
        if marker[0].lower() in ["brightfield", "bf"]:
            bf_index = marker[1]
            break
    if bf_index is None:
        print("Please define the brightfield channel index under MARKERS.")

    # Normalize to remove outliers (and help later UNet3D inference)  
    brightfield_norm = _normalize_percentile(lif_image[bf_index])

    # Remove out of focus brightfield haze using Difference of Gaussians (DoG)
    bf_dog = difference_of_gaussians(brightfield_norm, low_sigma, high_sigma)

    # Normalize DoG response to [0, 1] before inversion
    bf_dog = _normalize_full_range(bf_dog)

    # Invert black and white values to simulate fluorescently labelled cell boundaries
    bf_inv = 1 - bf_dog

    return bf_inv

sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


<Image layer 'PanSeg_UNet3D_input' at 0x1ed4f84c8e0>

In [78]:
# Predict root cell boundary probability maps
root_pmaps = predict_tiled_unet(
    raw=sim_fluo_cell_walls,
    input_layout="ZYX",
    model_dir=MODEL_DIR,
    patch=(80, 160, 160),
    patch_halo=(8, 16, 16),
    stride_ratio=0.75,
    batch_size=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_amp=True,
)

# root_pmaps: (C_out, Z, Y, X)
viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot\notebooks\..\src\utils\inference.py:163: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path, 

<Image layer 'root_unet_pmap' at 0x1ee04e7c9d0>

In [79]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_nuclei_labels_output_dir(RAW_DATA_DIRECTORY, lif_container_id)
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_nuclei_labels_if_available(nuclei_labels_dir, lif_image_name)

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Add the prediction to the napari viewer
viewer.add_labels(nuclei_labels)

Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_mock 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
Predictions already calculated for: Col0 3 ...loading


<Labels layer 'nuclei_labels' at 0x1ee09d9ef50>

In [80]:
import numpy as np
from scipy.ndimage import binary_fill_holes
from skimage.morphology import binary_closing, binary_erosion, disk

def generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=False):
    """
    Generate a coarse 3D root mask by combining predictions from a root boundary segmentation map 
    and nuclei label masks. This function includes slice-wise morphological closing and hole filling 
    to produce contiguous masks even when contours are broken.

    Args:
        root_pmaps (np.ndarray): Probability map of root boundaries (shape: [1, Z, Y, X]).
                                 The array should be indexed at [0] to obtain the 3D map.
        nuclei_labels (np.ndarray): 3D array of nuclei instance labels (same shape as ZYX).
        probability_threshold (float, optional): Threshold value for converting the root boundary probability map 
            to a binary mask. Voxels with a probability above this value are considered part of the root boundary.
            Default is 0.9. Higher values will result in a more conservative root boundary mask.
        visualize (bool, optional): If True, add relevant masks as layers to the global napari viewer. Default is False.

    Returns:
        np.ndarray: A boolean 3D mask (filled_2d_closed) representing the rough root segmentation.
    """
    # Threshold the root boundary probability map to create a binary mask
    root_boundary_mask = root_pmaps[0] > probability_threshold  # shape: (Z, Y, X)

    # Create a binary mask where nuclei are present
    nuclei_mask = nuclei_labels > 0  # shape: (Z, Y, X)

    # Combine the nuclei and root boundary masks as a starting point
    mask = (root_boundary_mask | nuclei_mask).astype(bool)

    # Preallocate the filled+closed output array
    filled_2d_closed = np.zeros_like(mask)

    # Perform slice-wise morphological closing and hole filling
    for z in range(mask.shape[0]):
        closed = binary_closing(mask[z], footprint=disk(5))           # Fill small holes/gaps in each slice
        filled_2d_closed[z] = binary_fill_holes(closed)               # Fill interior holes after closing

    # Optionally, visualize all relevant masks in napari
    if visualize:
        # Assumes that a global 'viewer' object exists
        viewer.add_image(root_boundary_mask, name="root_boundary_mask", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(nuclei_mask, name="nuclei_mask", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(mask, name="combined_mask", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(filled_2d_closed, name="closed+filled_2d_slice", colormap="yellow", blending="additive", opacity=0.6)

    return filled_2d_closed

In [81]:
# Generate a rough 3D root mask
filled_2d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)

In [82]:
def fill_root_3d(filled_2d_closed, occupancy_threshold=0.9, visualize=False):
    """
    Refine a preliminary 3D root mask by filling the root core based on slice-wise voxel occupancy,
    with additional erosion to ensure core robustness. The method preserves edges while solidifying the interior.

    Args:
        filled_2d_closed (np.ndarray): Boolean 3D array (Z, Y, X), preliminary root mask.
        occupancy_threshold (float, optional): Ratio (0-1). Voxels present in this fraction of slices belong to the core. Default is 0.9.
        visualize (bool, optional): Whether to add all relevant masks/layers to the global napari viewer.

    Returns:
        np.ndarray: Boolean 3D mask representing the refined and filled root mask (filled_final).
    """
    Z = filled_2d_closed.shape[0]

    # Compute per-pixel occupancy across Z
    occupancy = np.sum(filled_2d_closed, axis=0)  # shape: (Y, X)
    occupancy_norm = occupancy / Z

    # Threshold occupancy to define the 2D core present in sufficient slices
    threshold = int(np.ceil(occupancy_threshold * Z))
    core_2d = occupancy >= threshold

    # Erode the core to avoid overfilling edges
    core_2d_eroded = binary_erosion(core_2d, footprint=disk(3))

    # Expand eroded core back to 3D
    core_3d = np.repeat(core_2d_eroded[np.newaxis, ...], Z, axis=0)

    # Combine with the original to fill the core while preserving original edges
    filled_final = filled_2d_closed | core_3d

    # Optional napari visualization
    if visualize:
        viewer.add_image(occupancy_norm, name="occupancy_map")
        viewer.add_image(core_2d, name="core_2d", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(core_2d_eroded, name="core_2d_eroded", colormap="gray", blending="additive", opacity=1)
        viewer.add_image(filled_final, name="filled_final", colormap="green", blending="additive", opacity=0.5)
        
    return filled_final


In [83]:
# Refine and fill the rough 3D root mask
filled_root_3d = fill_root_3d(filled_2d_closed, occupancy_threshold=0.9, visualize=True)

In [84]:
from skimage.morphology import ball

filled_final_smooth = binary_erosion(filled_root_3d, footprint=ball(5))
viewer.add_image(filled_final_smooth, opacity=0.35)

<Image layer 'filled_final_smooth' at 0x1ee1a6da8c0>

In [85]:
from skimage.morphology import binary_closing, binary_opening, ball

smooth_mask = binary_closing(filled_final_smooth, ball(3))
smooth_mask = binary_opening(smooth_mask, ball(3))

viewer.add_image(smooth_mask, opacity=0.35)

<Image layer 'smooth_mask' at 0x1ee1d8b6770>

In [86]:
import numpy as np
from scipy.ndimage import distance_transform_edt
from skimage.measure import regionprops

# --------------------------------------------------
# Inputs
# --------------------------------------------------
labels = nuclei_labels
mask = smooth_mask.astype(bool)

# --------------------------------------------------
# 1. Distance to outside (boundary)
# --------------------------------------------------
# Pad mask with 1 voxel of zeros in all directions
mask_padded = np.pad(smooth_mask.astype(bool), pad_width=1, mode='constant', constant_values=0)

# Distance transform on padded mask
dist_padded = distance_transform_edt(mask_padded)

# Remove padding to restore original shape
dist_map = dist_padded[1:-1, 1:-1, 1:-1]

# --------------------------------------------------
# 2. Normalize by local thickness
# (approximate thickness = max distance in mask)
# --------------------------------------------------
max_dist = dist_map.max()

# Avoid division issues
if max_dist == 0:
    raise ValueError("Distance map is empty. Check mask.")

dist_norm = dist_map / max_dist

# --------------------------------------------------
# 3. Assign depth using centroid of each nucleus
# --------------------------------------------------
depth_per_label = {}

props = regionprops(labels)

for prop in props:
    label_id = prop.label
    
    # Centroid coordinates
    z, y, x = map(int, prop.centroid)
    
    depth = dist_norm[z, y, x]
    depth_per_label[label_id] = depth

# --------------------------------------------------
# 4. Map depth back to full image
# --------------------------------------------------
depth_image = np.zeros_like(dist_map, dtype=float)

for label_id, depth in depth_per_label.items():
    depth_image[labels == label_id] = depth

# --------------------------------------------------
# 5. Optional contrast enhancement
# --------------------------------------------------
nonzero = depth_image > 0
if np.any(nonzero):
    p1, p99 = np.percentile(depth_image[nonzero], (1, 99))
    depth_image = np.clip(depth_image, p1, p99)

# Re-normalize after clipping
depth_image = (depth_image - depth_image.min()) / (
    depth_image.max() - depth_image.min() + 1e-8
)

# --------------------------------------------------
# 6. Napari visualization
# --------------------------------------------------
viewer.add_labels(labels, name="nuclei_labels")

viewer.add_image(
    depth_image,
    name="nuclei_depth_normalized",
    colormap="viridis",
    blending="additive"
)

<Image layer 'nuclei_depth_normalized' at 0x1ee1d9a9e70>